# Prueba tecnica ENKI

## Paso 1: Extracción de datos con Python

### 1.1 Consumo de la API

In [1]:
import requests

url_openmeteo = "https://api.open-meteo.com/v1/forecast"

end_point = "?latitude=19.43&longitude=-99.13&hourly=temperature_2m,precipitation&past_days=7&forecast_days=0"

response = requests.get(url=url_openmeteo + end_point)

if response.status_code == 200:
    data = response.json()
    print("Solicitud exitosa. Datos obtenidos correctamente.")
    
else:
    raise Exception(f"Error en la API: {response.status_code}")

Solicitud exitosa. Datos obtenidos correctamente.


In [2]:
print(data)

{'latitude': 19.437609, 'longitude': -99.10715, 'generationtime_ms': 13.23390007019043, 'utc_offset_seconds': 0, 'timezone': 'GMT', 'timezone_abbreviation': 'GMT', 'elevation': 2238.0, 'hourly_units': {'time': 'iso8601', 'temperature_2m': '°C', 'precipitation': 'mm'}, 'hourly': {'time': ['2026-04-18T00:00', '2026-04-18T01:00', '2026-04-18T02:00', '2026-04-18T03:00', '2026-04-18T04:00', '2026-04-18T05:00', '2026-04-18T06:00', '2026-04-18T07:00', '2026-04-18T08:00', '2026-04-18T09:00', '2026-04-18T10:00', '2026-04-18T11:00', '2026-04-18T12:00', '2026-04-18T13:00', '2026-04-18T14:00', '2026-04-18T15:00', '2026-04-18T16:00', '2026-04-18T17:00', '2026-04-18T18:00', '2026-04-18T19:00', '2026-04-18T20:00', '2026-04-18T21:00', '2026-04-18T22:00', '2026-04-18T23:00', '2026-04-19T00:00', '2026-04-19T01:00', '2026-04-19T02:00', '2026-04-19T03:00', '2026-04-19T04:00', '2026-04-19T05:00', '2026-04-19T06:00', '2026-04-19T07:00', '2026-04-19T08:00', '2026-04-19T09:00', '2026-04-19T10:00', '2026-04-19

In [3]:
hourly = data.get("hourly", {})

time = hourly.get("time", [])
temperature = hourly.get("temperature_2m", [])
precipitation = hourly.get("precipitation", [])

In [4]:
records = []

for t, temp, prec in zip(time, temperature, precipitation):
    records.append({
        "time": t,
        "temperature_2m": temp,
        "precipitation": prec
    })

In [5]:
records

[{'time': '2026-04-18T00:00', 'temperature_2m': 28.5, 'precipitation': 0.0},
 {'time': '2026-04-18T01:00', 'temperature_2m': 25.8, 'precipitation': 0.0},
 {'time': '2026-04-18T02:00', 'temperature_2m': 24.6, 'precipitation': 0.0},
 {'time': '2026-04-18T03:00', 'temperature_2m': 23.5, 'precipitation': 0.0},
 {'time': '2026-04-18T04:00', 'temperature_2m': 22.9, 'precipitation': 0.0},
 {'time': '2026-04-18T05:00', 'temperature_2m': 22.5, 'precipitation': 0.0},
 {'time': '2026-04-18T06:00', 'temperature_2m': 21.0, 'precipitation': 0.0},
 {'time': '2026-04-18T07:00', 'temperature_2m': 20.6, 'precipitation': 0.0},
 {'time': '2026-04-18T08:00', 'temperature_2m': 19.7, 'precipitation': 0.0},
 {'time': '2026-04-18T09:00', 'temperature_2m': 18.4, 'precipitation': 0.0},
 {'time': '2026-04-18T10:00', 'temperature_2m': 17.3, 'precipitation': 0.0},
 {'time': '2026-04-18T11:00', 'temperature_2m': 16.7, 'precipitation': 0.0},
 {'time': '2026-04-18T12:00', 'temperature_2m': 16.1, 'precipitation': 0.0},

### 1.2 Limpieza y transformación de datos

In [6]:
import pandas as pd 

df = pd.DataFrame(records)
df['time'] = pd.to_datetime(df["time"])
df.columns = ["fecha", "temperatura_c", "precipitacion_mm"]
df.head()

,fecha,temperatura_c,precipitacion_mm
0,2026-04-18 00:00:00,28.5,0.0
1,2026-04-18 01:00:00,25.8,0.0
2,2026-04-18 02:00:00,24.6,0.0
3,2026-04-18 03:00:00,23.5,0.0
4,2026-04-18 04:00:00,22.9,0.0


In [7]:
df = df[df["fecha"].dt.hour.between(6,22)]

In [8]:
df.set_index("fecha", inplace=True)

In [9]:
# valores nulos 
df.isnull().sum()

temperatura_c       0
precipitacion_mm    0
dtype: int64

In [10]:
df[(df["temperatura_c"] < 0) | (df["precipitacion_mm"] < 0) ] 

,temperatura_c,precipitacion_mm
fecha,,


In [11]:
df.to_csv("datos_clima_cdmx.csv")

## Paso 2: Análisis con SQL

In [12]:
import sqlite3
import csv

conn = sqlite3.connect("base_ENKI.db")
cursor = conn.cursor()

cursor.execute("""CREATE TABLE IF NOT EXISTS clima_cdmx (
               fecha TIMESTAP,
               temperatura_c NUMERIC,
               precipitacion_mm NUMERIC,
               PRIMARY KEY (fecha)
               ) """)

with open("datos_clima_cdmx.csv", newline='', encoding='utf-8') as archivo:
    reader = csv.reader(archivo)
    
    next(reader)  # saltar encabezado
    
    for fila in reader:
        cursor.execute(
            "INSERT INTO clima_cdmx (fecha, temperatura_c, precipitacion_mm) VALUES (?, ?, ?)", 
            fila
        )

conn.commit()
conn.close()

In [13]:
def run_query(query, db="base_ENKI.db"):
        
    conn = sqlite3.connect(db)
    cursor = conn.cursor()
    
    cursor.execute(query)
    rows = cursor.fetchall()
    
    for row in rows:
        print(row)

    conn.close()

### Consulta A — Temperatura promedio por día

In [14]:
query1 = """SELECT 
                DATE(fecha) AS fecha,
                AVG(temperatura_c) AS avg_temperatura_c
            FROM clima_cdmx
            GROUP BY 1
            ORDER BY 2 DESC
        """

run_query(query1)


('2026-04-18', 21.446551724137926)
('2026-04-17', 20.54878048780488)
('2026-04-16', 19.71463414634146)
('2026-04-24', 19.323529411764707)
('2026-04-19', 19.15000000000001)
('2026-04-22', 18.30344827586207)
('2026-04-21', 18.18448275862069)
('2026-04-23', 18.04705882352941)
('2026-04-20', 17.362068965517242)


### Consulta B — Horas con precipitación

In [15]:
query2 = """SELECT 
                fecha,
                precipitacion_mm
            FROM clima_cdmx
            WHERE precipitacion_mm > 0 
            ORDER BY 1 
        """

run_query(query2)

('2026-04-16 02:00:00', 1.1)
('2026-04-16 03:00:00', 0.4)
('2026-04-20 18:00:00', 0.1)
('2026-04-20 22:00:00', 0.2)
('2026-04-20 22:00:00', 0.2)
('2026-04-21 00:00:00', 0.9)
('2026-04-21 01:00:00', 2.5)
('2026-04-21 02:00:00', 1.3)
('2026-04-21 03:00:00', 0.7)
('2026-04-21 04:00:00', 0.6)
('2026-04-21 05:00:00', 0.2)
('2026-04-21 06:00:00', 0.6)
('2026-04-21 06:00:00', 0.6)
('2026-04-21 10:00:00', 0.2)
('2026-04-21 11:00:00', 0.2)
('2026-04-21 22:00:00', 0.1)
('2026-04-21 22:00:00', 0.1)
('2026-04-21 23:00:00', 0.4)
('2026-04-22 00:00:00', 0.1)
('2026-04-22 01:00:00', 0.5)
('2026-04-22 02:00:00', 0.7)
('2026-04-22 03:00:00', 1.5)
('2026-04-22 20:00:00', 0.1)
('2026-04-22 20:00:00', 0.1)
('2026-04-22 21:00:00', 0.2)
('2026-04-22 21:00:00', 0.2)
('2026-04-22 22:00:00', 0.1)
('2026-04-22 23:00:00', 0.9)
('2026-04-23 22:00:00', 0.1)
('2026-04-24 22:00:00', 1.6)


### Consulta C — Día con mayor variación térmica

In [16]:
query3 = """
         SELECT 
            DATE(fecha),
            MAX(temperatura_c) AS temperatura_max,
            MIN(temperatura_c) AS temperatura_min,
            MAX(temperatura_c) - MIN(temperatura_c) AS variacion_termica
         FROM clima_cdmx
         GROUP BY 1
         ORDER BY 4 DESC
         LIMIT 1
            
         """

run_query(query3)

('2026-04-18', 30.1, 13.9, 16.200000000000003)


### Consulta D — Resumen diario

In [17]:
query4 = """
            SELECT 
                DATE(fecha) AS dia,
                MIN(temperatura_c) AS min_temperatura_c,
                MAX(temperatura_c) AS max_temperatura_c,
                AVG(temperatura_c) AS avg_temperatura_c,
                SUM(precipitacion_mm) AS precipitacion_acumulada_mm
            FROM clima_cdmx
            GROUP BY 1
            ORDER BY 1
        """

run_query(query4)

('2026-04-16', 13.2, 26.8, 19.71463414634146, 1.5)
('2026-04-17', 14.1, 27.6, 20.54878048780488, 0)
('2026-04-18', 13.9, 30.1, 21.446551724137926, 0)
('2026-04-19', 12.9, 27.5, 19.15000000000001, 0)
('2026-04-20', 12.8, 24.3, 17.362068965517242, 0.5)
('2026-04-21', 13.8, 25.9, 18.18448275862069, 8.399999999999999)
('2026-04-22', 13, 25.1, 18.30344827586207, 4.3999999999999995)
('2026-04-23', 12.2, 25.5, 18.04705882352941, 0.1)
('2026-04-24', 12.6, 28.7, 19.323529411764707, 1.6)


## Paso 3: Documentación Técnica

1. Flujo del pipeline.

El pipeline realiza la extracción de datos climáticos desde la API de Open-Meteo utilizando requests. 
Posteriormente, los datos son transformados con pandas: se convierten a formato datetime, se renombran las columnas y se filtran los registros entre las 06:00 y las 22:00 horas. 
Se aplican validaciones básicas para identificar valores nulos y negativos en las variables numéricas. 
Finalmente, los datos limpios se exportan a un archivo CSV y se cargan en una base de datos  para su posterior consulta.

2. Decisiones Tecnicas.

Se utilizó la librería pandas para la transformación de datos debido a su eficiencia en el manejo de estructuras tabulares y operaciones de limpieza. 
Para el consumo de la API se empleó requests, ya que es una solución simple y ampliamente utilizada para llamadas HTTP en Python. 
Se implementó manejo de errores con try/except para controlar fallos en la conexión o respuestas inválidas de la API. 

3. Mejoras futuras 

Si tuviera más tiempo, implementaría un manejo más robusto de errores y validaciones, incluyendo alertas automáticas en caso de fallos. 
También integraría el pipeline en un scheduler (como cron o Airflow) para su ejecución periódica. 
Finalmente, consideraría almacenar los datos en una base de datos más escalable (como PostgreSQL) y agregar pruebas unitarias para garantizar la calidad del código.